## 2. 웹스크래핑 연습문제

2-1. Nate 뉴스기사 제목 스크래핑하기 (필수) 

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import Image, display

url = 'https://news.nate.com/recent?mid=n0100'
print(url)

req_header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/104.0.0.0 Safari/537.36'
}

res = requests.get(url, headers=req_header)
print(type(res))
print(res.status_code)

if res.ok:
    res.encoding = 'euc-kr'
    html = res.text
    soup = BeautifulSoup(html, 'html.parser')
    
    tags = soup.select("div.postListType.noListTitle div.mlt01")

    print(len(tags))

    for idx,div_tag in enumerate(tags,1): 
        print(f'============>> {idx}')       
        a_tag = div_tag.find('a')
        a_join_url = urljoin(url,a_tag['href'])
        print(a_join_url)
        
        #이미지 출력         
        img_tag = div_tag.select_one('span.ib img')
        if img_tag:
            photo_url = urljoin(url,img_tag['src'])
            # 이미지의 상대경로 앞에 절대 경로인 url을 합쳐서 절대 경로로 바꿈.
            print(photo_url)
            display(Image(url=photo_url))
        
        #뉴스 기사 출력
        h2_tag = div_tag.select_one('span.tb h2.tit')
        title = h2_tag.text
        print(title)

else:
    print(f'에러코드 = {res.status_code}')

## 2-2. 하나의 네이버 웹툰과 1개의 회차에 대한 Image 다운로드 하기 (필수) 

In [15]:
import os
import requests
from bs4 import BeautifulSoup

def download_one_episode(title, no, url):
    # 1. 디렉토리 생성 (img\제목\회차번호)
    save_path = os.path.join('img', title, str(no))
    os.makedirs(save_path, exist_ok=True)
    
    # 2. 웹툰 페이지 요청 헤더 설정 
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': url
    }

    res = requests.get(url, headers=headers)
    if not res.ok:
        print(f"웹툰 페이지 접속 실패: {res.status_code}")
        return

    soup = BeautifulSoup(res.text, 'html.parser')

    # 3. 웹툰 이미지 태그들 찾기
    img_tags = soup.select("img[src*='IMAG01']")
    
    if not img_tags:
        print("이미지를 찾을 수 없습니다. 선택자를 확인해 주세요.")
        return

    print(f"[{title}] {no}화 다운로드 시작...")

    # 4. 이미지 다운로드 및 저장
    for i, img_tag in enumerate(img_tags):
        img_url = img_tag['src']
        
        # 이미지 데이터 요청
        img_res = requests.get(img_url, headers=headers)
        
        if img_res.ok:
            img_data = img_res.content
            # 파일명 생성 
            file_name = f"{i:03d}_{os.path.basename(img_url.split('?')[0])}"
            file_full_path = os.path.join(save_path, file_name)
            
            # 바이너리 모드로 저장
            with open(file_full_path, 'wb') as f:
                f.write(img_data)
            print(f"저장 완료: {file_name} ({len(img_data):,} bytes)")
        else:
            print(f"이미지 다운로드 실패: {img_url}")

    print(f"\n✅ '{title}' {no}화 모든 이미지 다운로드 완료!")

# 함수 호출 예시
download_one_episode('괴물 천재선수들이 날 너무 좋아함', 9, 'https://comic.naver.com/webtoon/detail?titleId=843901&no=9&week=sun')

[괴물 천재선수들이 날 너무 좋아함] 9화 다운로드 시작...
저장 완료: 000_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_1.jpg (255,530 bytes)
저장 완료: 001_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_2.jpg (203,567 bytes)
저장 완료: 002_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_3.jpg (174,585 bytes)
저장 완료: 003_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_4.jpg (227,371 bytes)
저장 완료: 004_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_5.jpg (274,280 bytes)
저장 완료: 005_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_6.jpg (176,032 bytes)
저장 완료: 006_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_7.jpg (243,124 bytes)
저장 완료: 007_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_8.jpg (161,332 bytes)
저장 완료: 008_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_9.jpg (227,641 bytes)
저장 완료: 009_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_10.jpg (176,666 bytes)
저장 완료: 010_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_11.jpg (146,703 by

## 2-3. 하나의 네이버 웹툰과 여러개의 회차에 대한 Image 다운로드 하기 (선택)

In [7]:
import requests
from bs4 import BeautifulSoup
from pprint import pprint
from urllib.parse import urlparse, parse_qs

def download_all_episode(title,episode_url):
    # url을 파싱해서 titleId를 알아내기
    parsed_url = urlparse(episode_url)
    query_params = parse_qs(parsed_url.query)
    title_id = query_params.get('titleId', [''])[0]
    
    api_url = f'https://comic.naver.com/api/article/list?titleId={title_id}'
    res = requests.get(api_url)
    print(res.status_code)    
    if res.ok:
        #pprint(res.json()['articleList'])
        for article in res.json()['articleList']:
            no = article['no']
            detail_url = f'https://comic.naver.com/webtoon/detail?titleId={title_id}&no={no}'
            print(detail_url)
        

if __name__ == '__main__': 
    download_all_episode('삽가능','https://comic.naver.com/webtoon/list?titleId=838708')

200
https://comic.naver.com/webtoon/detail?titleId=838708&no=42
https://comic.naver.com/webtoon/detail?titleId=838708&no=41
https://comic.naver.com/webtoon/detail?titleId=838708&no=40
https://comic.naver.com/webtoon/detail?titleId=838708&no=39
https://comic.naver.com/webtoon/detail?titleId=838708&no=38
https://comic.naver.com/webtoon/detail?titleId=838708&no=37
https://comic.naver.com/webtoon/detail?titleId=838708&no=36
https://comic.naver.com/webtoon/detail?titleId=838708&no=35
https://comic.naver.com/webtoon/detail?titleId=838708&no=34
https://comic.naver.com/webtoon/detail?titleId=838708&no=33
https://comic.naver.com/webtoon/detail?titleId=838708&no=32
https://comic.naver.com/webtoon/detail?titleId=838708&no=31
https://comic.naver.com/webtoon/detail?titleId=838708&no=30
https://comic.naver.com/webtoon/detail?titleId=838708&no=29
https://comic.naver.com/webtoon/detail?titleId=838708&no=28
https://comic.naver.com/webtoon/detail?titleId=838708&no=27
https://comic.naver.com/webtoon/deta

In [8]:
from urllib.parse import urlparse, parse_qs

"""url에 titleId를 반환하는 함수"""
def get_title_id(url):
    parsed_url = urlparse(url)
    query_params = parse_qs(parsed_url.query)
    title_id = query_params.get('titleId', [''])[0]
    return title_id

#테스트 하기
url = 'https://comic.naver.com/webtoon/list?titleId=826419'
print(get_title_id(url))  # 출력: 826419

826419


In [9]:
def calculate_pages(total_items, items_per_page=20):
    """총 페이지 수 계산 함수"""
    return (total_items + items_per_page - 1) // items_per_page

# 예제 사용
total_items = 49
items_per_page = 20

total_pages = calculate_pages(total_items)
print(f"총 {total_items}개의 항목을 {items_per_page}개씩 출력할 때 필요한 페이지 수: {total_pages}")
# 출력: 총 49개의 항목을 20개씩 출력할 때 필요한 페이지 수: 3

총 49개의 항목을 20개씩 출력할 때 필요한 페이지 수: 3


In [10]:
import requests

def download_all_episode(title,episode_url):
    title_id = get_title_id(episode_url)

    ajax_url = f'https://comic.naver.com/api/article/list?titleId={title_id}'               
    res = requests.get(ajax_url)

    if res.ok:
        total_count = res.json()['totalCount']
        for count in range(calculate_pages(total_count)):
            # count 변수 0,1,2 // page 번호는 1,2,3
            page = count + 1
            req_param = { "page": page }
            print(req_param)
            res = requests.get(ajax_url, params=req_param)
            for article in res.json()['articleList']:
                no = article['no']
                detail_url = f'https://comic.naver.com/webtoon/detail?titleId={title_id}&no={no}'
                print(detail_url)
        

if __name__ == '__main__': 
    download_all_episode('삽가능','https://comic.naver.com/webtoon/list?titleId=838708')

{'page': 1}
https://comic.naver.com/webtoon/detail?titleId=838708&no=42
https://comic.naver.com/webtoon/detail?titleId=838708&no=41
https://comic.naver.com/webtoon/detail?titleId=838708&no=40
https://comic.naver.com/webtoon/detail?titleId=838708&no=39
https://comic.naver.com/webtoon/detail?titleId=838708&no=38
https://comic.naver.com/webtoon/detail?titleId=838708&no=37
https://comic.naver.com/webtoon/detail?titleId=838708&no=36
https://comic.naver.com/webtoon/detail?titleId=838708&no=35
https://comic.naver.com/webtoon/detail?titleId=838708&no=34
https://comic.naver.com/webtoon/detail?titleId=838708&no=33
https://comic.naver.com/webtoon/detail?titleId=838708&no=32
https://comic.naver.com/webtoon/detail?titleId=838708&no=31
https://comic.naver.com/webtoon/detail?titleId=838708&no=30
https://comic.naver.com/webtoon/detail?titleId=838708&no=29
https://comic.naver.com/webtoon/detail?titleId=838708&no=28
https://comic.naver.com/webtoon/detail?titleId=838708&no=27
https://comic.naver.com/webt

In [ ]:
import requests
from time import sleep

def download_all_episode(title,episode_url):
    title_id = get_title_id(episode_url)

    ajax_url = f'https://comic.naver.com/api/article/list?titleId={title_id}'               
    res = requests.get(ajax_url)

    if res.ok:
        total_count = res.json()['totalCount']
        for count in range(calculate_pages(total_count)):
            page = count + 1
            req_param = { "page": page}
            print(req_param)
            res = requests.get(ajax_url, params=req_param)
            for article in res.json()['articleList']:
                no = article['no']
                detail_url = f'https://comic.naver.com/webtoon/detail?titleId={title_id}&no={no}'
                print(detail_url)
                download_one_episode(title,no,detail_url)
                #0.5초간 프로세스를 중지함, 기계가 아니라 사람처럼 보이게 하려고
                sleep(0.5)

if __name__ == '__main__': 
    #download_all_episode('롤플레잉','https://comic.naver.com/webtoon/list?titleId=826419')
    download_all_episode('삽가능','https://comic.naver.com/webtoon/list?titleId=838708')

{'page': 1}
https://comic.naver.com/webtoon/detail?titleId=838708&no=42


NameError: name 'download_one_episode' is not defined